# 세그멘테이션: 노면 균열

균열이 가늘어서 박스로 잡으면 대부분 배경이다. 픽셀마다 균열인지를 칠한다.

**런타임 → 런타임 유형 변경 → GPU(T4)** 를 먼저 고른다.

zip에는 데이터와, 검출의 YOLO 시작 가중치가 있다. 학습·추론 코드는 이 노트북 셀에 있다.
`/content`는 세션이 끝나면 비므로 zip은 드라이브에 남겨 둔다.


## 1. 데이터 zip

드라이브에서 해당 zip을 `링크가 있는 모든 사용자`로 공유한 뒤, 링크를 `ZIP_URL`에 넣는다.

파일이 100MB보다 크면 `wget`이 구글 확인 페이지를 받는다. 드라이브 링크는 `gdown`, 직접 열리는 주소는 `wget`을 쓴다.


In [ ]:
ZIP_URL = "https://drive.google.com/file/d/1ooQt7A3VxUBlnfy0VNLr-4u74c42w620/view?usp=sharing"

TASK = "segmentation"
ZIP_PATH = "/content/03_segmentation.zip"
DST = f"/content/{TASK}"


## 2. 패키지


In [ ]:
!pip install -q gdown


## 3. zip 다운로드


In [ ]:
from pathlib import Path
import os


def fetch_zip(url, dest):
    url = (url or "").strip()
    if not url or "여기에" in url:
        raise SystemExit(
            "위 셀 ZIP_URL에 구글 드라이브 공유 링크(또는 직접 zip 주소)를 넣는다."
        )

    is_drive = "drive.google.com" in url or "docs.google.com" in url

    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 10_000:
        print("이미 받은 파일이 있다:", dest, dest.stat().st_size, "bytes")
        return

    dest.parent.mkdir(parents=True, exist_ok=True)

    os.environ["ZIP_URL"] = url
    os.environ["ZIP_PATH"] = str(dest)
    if is_drive:
        !gdown --fuzzy "$ZIP_URL" -O "$ZIP_PATH"
    else:
        !wget -c -O "$ZIP_PATH" "$ZIP_URL"

    print("받음", dest, dest.stat().st_size, "bytes")


fetch_zip(ZIP_URL, ZIP_PATH)


In [ ]:
!rm -rf {DST}


In [ ]:
!unzip -q -o {ZIP_PATH} -d /content


In [ ]:
from pathlib import Path
import torch
from IPython.display import display
from PIL import Image

ROOT = Path(DST)
if not ROOT.exists():
    raise SystemExit(f"{ROOT} 가 없다. zip 안에 과제 폴더가 있는지 확인한다.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT", ROOT)
print("torch", torch.__version__)
print("device", DEVICE)
if DEVICE.type == "cuda":
    print("gpu", torch.cuda.get_device_name(0))


def show_img(img, width=900):
    """셀 결과에 PIL 이미지를 띄운다. 너무 가로로 긴 장만 줄인다."""
    img = img.convert("RGB")
    w, h = img.size
    if w > width:
        img = img.resize((width, int(h * width / w)), Image.BILINEAR)
    display(img)


## 4. Bounding Box의 한계 도출

검출은 객체마다 직사각형 하나를 준다. 그 직사각형으로 ‘어디를 손댈지’를 정하면, 아래 두 경우에 틀린다.

1. **다른 물체가 박스 안에 들어간다.** 잡초에 제초제를 뿌리면 당근까지 죽는다. 아래 그림의 왼쪽은 원본, 가운데는 박스, 오른쪽은 마스크다. 이 잡초 장은 학습에 쓰지 않는다.
2. **물체가 가늘고 길면 박스가 장면을 거의 덮는다.** 이번 과제의 균열이 이 경우다. 강의용 세 장은 균열 픽셀이 화면의 1\~2%인데,마스크를 한 박스로 감싸면 박스가 화면의 80\~90%를 차지하고, 그 안의 약 98%는 멀쩡한 노면이다. 그 박스로 보수를 지시하면 금만 손보는 것이 아니라 포장을 거의 다 건드린다.

아래 숫자는 정답 마스크로 계산한다. 학습이 아니라, 박스가 왜 부족한지를 보기 위한 것이다.


In [ ]:
from PIL import Image

weed = ROOT / "examples" / "weed_box_vs_mask.jpg"
if weed.exists():
    show_img(Image.open(weed), width=980)
else:
    print("도입 그림이 없다:", weed)


## 5. 신경망 모델

`torch`와 `torchvision`에는 U-Net이 없다. 분류의 ResNet18과 달리 공식 가중치도 없다.
아래는 Ronneberger U-Net을 실습용으로 줄인 코드다. `print(model)`은 skip concat이 안 보이므로, `show_unet_stack`으로 텐서 모양을 본다.
이름에서 앞부분은 역할, 숫자는 깊이다. 1이 원본 해상도에 가깝고 4가 더 깊다. 같은 숫자는 서로 짝이다.
- `enc` = encoder. 내려가며 압축한다.
- `skipN` = `encN`의 출력. 나중에 같은 숫자의 디코더에 붙인다.
- `bottleneck` = U자 맨 아래, 가장 낮은 해상도.
- `upN` = upsample. 해상도를 두 배로 키워 `skipN`과 맞춘다.
- `mergedN` = `upN`과 `skipN`을 채널 방향으로 이어 붙인 텐서.
- `decN` = decoder. 붙인 특징을 다시 합성곱한다.


In [ ]:
import torch
from torch import nn


class DoubleConv(nn.Module):
    """같은 해상도에서 합성곱을 두 번 쌓는다."""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        # enc = encoder. 해상도를 줄이며 채널을 늘린다. 1이 넓고 4가 깊다.
        self.enc1 = DoubleConv(in_ch, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.enc4 = DoubleConv(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 8, base * 16)
        # upN = upsample. encN·skipN과 같은 해상도로 키운다.
        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        # decN = decoder. upN과 skipN을 붙인 뒤 다시 합성곱한다.
        self.dec4 = DoubleConv(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        skip1 = self.enc1(x)
        skip2 = self.enc2(self.pool(skip1))
        skip3 = self.enc3(self.pool(skip2))
        skip4 = self.enc4(self.pool(skip3))
        bottleneck = self.bottleneck(self.pool(skip4))

        up4 = self.up4(bottleneck)
        merged4 = torch.cat([up4, skip4], dim=1)
        dec4 = self.dec4(merged4)

        up3 = self.up3(dec4)
        merged3 = torch.cat([up3, skip3], dim=1)
        dec3 = self.dec3(merged3)

        up2 = self.up2(dec3)
        merged2 = torch.cat([up2, skip2], dim=1)
        dec2 = self.dec2(merged2)

        up1 = self.up1(dec2)
        merged1 = torch.cat([up1, skip1], dim=1)
        dec1 = self.dec1(merged1)
        return self.head(dec1)


def show_unet_stack(model, h=384, w=544):
    """더미 한 장을 통과시킨 뒤, U자 구조와 skip concat을 그림으로 그린다."""
    from io import BytesIO

    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    from PIL import Image

    model.eval()
    x = torch.zeros(1, 3, h, w)
    with torch.no_grad():
        skip1 = model.enc1(x)
        skip2 = model.enc2(model.pool(skip1))
        skip3 = model.enc3(model.pool(skip2))
        skip4 = model.enc4(model.pool(skip3))
        bottleneck = model.bottleneck(model.pool(skip4))
        dec4 = model.dec4(torch.cat([model.up4(bottleneck), skip4], dim=1))
        dec3 = model.dec3(torch.cat([model.up3(dec4), skip3], dim=1))
        dec2 = model.dec2(torch.cat([model.up2(dec3), skip2], dim=1))
        dec1 = model.dec1(torch.cat([model.up1(dec2), skip1], dim=1))
        out = model.head(dec1)

    def label(t):
        _, c, hh, ww = t.shape
        return f"{c} ch {hh}×{ww}"

    fig, ax = plt.subplots(figsize=(11.2, 8.2))
    ax.set_xlim(0, 10)
    ax.set_ylim(0.15, 11.35)
    ax.axis("off")
    ax.set_title("U-Net   dashed arrow = skip concat", fontsize=14, pad=8)

    boxes = {}

    def box(key, cx, cy, title, tensor, color):
        bw, bh = 1.85, 1.12
        patch = FancyBboxPatch(
            (cx - bw / 2, cy - bh / 2),
            bw,
            bh,
            boxstyle="round,pad=0.02,rounding_size=0.1",
            facecolor=color,
            edgecolor="#222",
            linewidth=1.2,
        )
        ax.add_patch(patch)
        ax.text(cx, cy + 0.22, title, ha="center", va="center", fontsize=9, fontweight="bold", color="#111")
        ax.text(cx, cy - 0.22, label(tensor), ha="center", va="center", fontsize=8, color="#222")
        boxes[key] = (cx, cy, bw, bh)

    def down(src, dst, text):
        x1, y1, _, h1 = boxes[src]
        x2, y2, _, h2 = boxes[dst]
        ax.annotate(
            "",
            xy=(x2, y2 + h2 / 2 + 0.02),
            xytext=(x1, y1 - h1 / 2 - 0.02),
            arrowprops=dict(arrowstyle="-|>", color="#333", lw=1.4, mutation_scale=10),
        )
        ax.text((x1 + x2) / 2 - 0.08, (y1 + y2) / 2, text, fontsize=7, color="#444", ha="right", va="center")

    def skip(src, dst):
        x1, y1, w1, _ = boxes[src]
        x2, y2, w2, _ = boxes[dst]
        ax.annotate(
            "",
            xy=(x2 - w2 / 2 - 0.02, y2),
            xytext=(x1 + w1 / 2 + 0.02, y1),
            arrowprops=dict(
                arrowstyle="-|>",
                color="#d06000",
                lw=1.6,
                ls=(0, (5, 3)),
                mutation_scale=10,
            ),
        )
        ax.text((x1 + x2) / 2, y1 + 0.32, "skip concat", ha="center", fontsize=7.5, color="#d06000")

    box("in", 2.0, 10.35, "input", x, "#ececec")
    box("e1", 2.0, 8.75, "enc1  →  skip1", skip1, "#9ec0ea")
    box("e2", 2.0, 7.15, "enc2  →  skip2", skip2, "#7aa8de")
    box("e3", 2.0, 5.55, "enc3  →  skip3", skip3, "#568fd0")
    box("e4", 2.0, 3.95, "enc4  →  skip4", skip4, "#3a74b8")
    box("bot", 5.0, 1.55, "bottleneck", bottleneck, "#4a4a6a")
    box("d4", 8.0, 3.95, "dec4", dec4, "#2f9e66")
    box("d3", 8.0, 5.55, "dec3", dec3, "#45b57a")
    box("d2", 8.0, 7.15, "dec2", dec2, "#63c890")
    box("d1", 8.0, 8.75, "dec1", dec1, "#86d8ac")
    box("head", 8.0, 10.35, "head  (mask)", out, "#ececec")

    down("in", "e1", "")
    down("e1", "e2", "MaxPool 1/2")
    down("e2", "e3", "MaxPool 1/2")
    down("e3", "e4", "MaxPool 1/2")

    x1, y1, _, h1 = boxes["e4"]
    x2, y2, _, h2 = boxes["bot"]
    ax.annotate(
        "",
        xy=(x2 - 0.95, y2 + h2 / 2),
        xytext=(x1 + 0.2, y1 - h1 / 2),
        arrowprops=dict(arrowstyle="-|>", color="#333", lw=1.4, mutation_scale=10),
    )
    ax.text(3.15, 2.55, "MaxPool 1/2", fontsize=7, color="#444")
    x3, y3, _, h3 = boxes["d4"]
    ax.annotate(
        "",
        xy=(x3 - 0.2, y3 - h3 / 2),
        xytext=(x2 + 0.95, y2 + h2 / 2),
        arrowprops=dict(arrowstyle="-|>", color="#333", lw=1.4, mutation_scale=10),
    )
    ax.text(6.55, 2.55, "up4", fontsize=7, color="#444")

    for src, dst, text in (("d4", "d3", "up3"), ("d3", "d2", "up2"), ("d2", "d1", "up1"), ("d1", "head", "")):
        x1, y1, _, h1 = boxes[src]
        x2, y2, _, h2 = boxes[dst]
        ax.annotate(
            "",
            xy=(x2, y2 - h2 / 2 - 0.02),
            xytext=(x1, y1 + h1 / 2 + 0.02),
            arrowprops=dict(arrowstyle="-|>", color="#333", lw=1.4, mutation_scale=10),
        )
        if text:
            ax.text(x1 + 0.12, (y1 + y2) / 2, text, fontsize=7, color="#444", ha="left", va="center")

    skip("e1", "d1")
    skip("e2", "d2")
    skip("e3", "d3")
    skip("e4", "d4")

    n = sum(p.numel() for p in model.parameters())
    ax.text(5.0, 0.42, f"params {n:,}", ha="center", fontsize=9)

    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=140, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    buf.seek(0)
    show_img(Image.open(buf), width=980)


show_unet_stack(UNet())

## 6. 학습

손실은 BCE와 Dice를 더한다. 가느다란 균열은 Dice가 없으면 잘 안 잡힌다.


In [ ]:
import json
import random

import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms.functional import to_tensor

EPOCHS = 25
BATCH = 12
WIDTH, HEIGHT = 544, 384
LR = 1e-3
SEED = 42


class CrackDataset(Dataset):
    def __init__(self, img_dir, lab_dir, size):
        self.images = sorted(img_dir.glob("*.jpg"))
        if not self.images:
            raise FileNotFoundError(img_dir)
        self.lab_dir = lab_dir
        self.size = size

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_p = self.images[idx]
        lab_p = self.lab_dir / f"{img_p.stem}.png"
        img = Image.open(img_p).convert("RGB").resize(self.size, Image.BILINEAR)
        mask = Image.open(lab_p).convert("L").resize(self.size, Image.NEAREST)
        x = to_tensor(img)
        y = (to_tensor(mask) > 0.5).float()
        return x, y, img_p.name


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def bce_dice(logits, target):
    bce = F.binary_cross_entropy_with_logits(logits, target)
    p = logits.sigmoid()
    num = 2 * (p * target).sum(dim=(0, 2, 3))
    den = p.sum(dim=(0, 2, 3)) + target.sum(dim=(0, 2, 3))
    dice = 1 - (num + 1.0) / (den + 1.0)
    return bce + dice.mean()


@torch.no_grad()
def eval_metrics(model, loader):
    model.eval()
    inter = union = dice_num = dice_den = 0.0
    for x, y, _ in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = (model(x).sigmoid() > 0.5).float()
        inter += (pred * y).sum().item()
        union += ((pred + y) > 0).float().sum().item()
        dice_num += 2 * (pred * y).sum().item()
        dice_den += pred.sum().item() + y.sum().item()

    return {
        "iou": inter / max(union, 1.0),
        "dice": dice_num / max(dice_den, 1.0),
    }


set_seed(SEED)
size = (WIDTH, HEIGHT)
data = ROOT / "dataset"
runs = ROOT / "runs"
train_ds = CrackDataset(data / "train_img", data / "train_lab", size)
val_ds = CrackDataset(data / "test_img", data / "test_lab", size)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=0)

model = UNet().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
runs.mkdir(parents=True, exist_ok=True)
best_dice = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    n = 0
    for x, y, _ in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = bce_dice(model(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item() * x.size(0)
        n += x.size(0)

    metrics = eval_metrics(model, val_loader)
    print(
        f"epoch {epoch:03d}  loss {running / n:.4f}  "
        f"val dice {metrics['dice']:.3f} iou {metrics['iou']:.3f}"
    )
    if metrics["dice"] >= best_dice:
        best_dice = metrics["dice"]
        torch.save({"model": model.state_dict(), "size": size}, runs / "best.pt")

print("saved", runs / "best.pt", "best_val_dice", round(best_dice, 4))


## 7. 추론 및 시각화

왼쪽부터 입력, 정답 마스크, U-Net 예측, 예측을 한 박스로 근사한 결과다.
앞에서 정답 마스크로 본 것과 같이, 예측을 한 박스로 감싸도 박스 안 대부분이 배경이다.
마스크 윤곽 좌표는 길어서 출력하지 않는다. 균열 픽셀 수와 박스 안 배경 비율, Dice만 적는다.

In [ ]:
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont


def font(size):
    for path in (
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    ):
        try:
            return ImageFont.truetype(path, size)
        except OSError:
            continue
    return ImageFont.load_default()


def mask_aabb(mask):
    ys, xs = np.where(mask > 0)
    if xs.size == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())


def blend(img, mask, color, alpha):
    overlay = Image.new("RGB", img.size, color)
    m = Image.fromarray((mask * 255).astype(np.uint8), mode="L")
    return Image.composite(Image.blend(img, overlay, alpha), img, m)


def caption(img, text):
    bar_h = 28
    canvas = Image.new("RGB", (img.width, img.height + bar_h), (20, 20, 20))
    canvas.paste(img, (0, bar_h))
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 6), text, fill=(255, 255, 255), font=font(16))
    return canvas


weights = ROOT / "runs" / "best.pt"
ckpt = torch.load(weights, map_location="cpu", weights_only=True)
size = tuple(ckpt.get("size", (544, 384)))

model = UNet()
model.load_state_dict(ckpt["model"])
model.eval()

images = sorted((ROOT / "examples").glob("[0-9][0-9].jpg"))

for path in images:
    raw = Image.open(path).convert("RGB")
    img = raw.resize(size, Image.BILINEAR)
    x = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
    with torch.no_grad():
        pred = (model(x.unsqueeze(0)).sigmoid()[0, 0] > 0.5).cpu().numpy()

    pred_ov = blend(img, pred, (40, 200, 80), 0.5)
    boxed = pred_ov.copy()
    box = mask_aabb(pred)
    if box:
        ImageDraw.Draw(boxed).rectangle(box, outline=(220, 40, 40), width=4)

    mask_path = path.parent / f"{path.stem}_mask.png"
    gt = None
    if mask_path.exists():
        gt = np.array(Image.open(mask_path).convert("L").resize(size, Image.NEAREST)) > 127
        gt_panel = blend(img, gt, (40, 140, 255), 0.5)
    else:
        gt_panel = img.copy()

    panels = [
        caption(img, "input"),
        caption(gt_panel, "GT mask" if gt is not None else "GT missing"),
        caption(pred_ov, "U-Net pred"),
        caption(boxed, "pred as one box"),
    ]
    w, h = panels[0].size
    canvas = Image.new("RGB", (w * 4, h))
    for i, panel in enumerate(panels):
        canvas.paste(panel, (i * w, 0))

    crack_px = int(pred.sum())
    box_px = 0 if box is None else (box[2] - box[0] + 1) * (box[3] - box[1] + 1)
    print(path.name)
    print(f"  균열 {crack_px} px")
    if box_px:
        print(f"  박스 {box_px} px  (배경 {1 - crack_px / box_px:.1%})")
    if gt is not None:
        inter = float((pred & gt).sum())
        den = float(pred.sum() + gt.sum())
        dice = (2 * inter / den) if den else 0.0
        print(f"  dice {dice:.3f}")

    show_img(canvas, width=980)
    print()
